# ⏳ Voice Aging System MVP — Qwen3-TTS

This notebook explores the creation of voice aging progressions for characters using Qwen3-TTS. This is highly useful for games featuring age-progression storylines, flashbacks, or different chronological versions of player characters.

In [ ]:
# Install dependencies
!pip install -q qwen-tts soundfile

In [ ]:
import os
import gc
import torch
import soundfile as sf
import numpy as np
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

OUTPUT_DIR = "/content/voice_aging"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

In [ ]:
CHARACTERS = {
    "warrior": {
        "name": "Aldric",
        "personality": "brave, direct, passionate about honor",
        "ages": {
            "young": {"age": "17", "voice_mod": "teenage voice, slightly cracking with adolescent energy, eager and impulsive"},
            "adult": {"age": "38", "voice_mod": "peak physical form, confident resonance, battle-tested authority"},
            "elder": {"age": "72", "voice_mod": "gravelly, slower, wisdom replacing vigor, slight breathiness"}
        },
        "lines": [
            "I will prove myself on the battlefield — watch me.",
            "After twenty years of war, I know what victory truly costs.",
            "My sword arm is weaker now, but my mind holds every lesson learned in blood."
        ]
    },
    "scholar": {
        "name": "Mira",
        "personality": "curious, precise, intellectually driven",
        "ages": {
            "young": {"age": "19", "voice_mod": "bright, quick, slightly nervous energy, overly eager to prove intelligence"},
            "adult": {"age": "41", "voice_mod": "poised, methodical, comfortable authority in her domain"},
            "elder": {"age": "78", "voice_mod": "still sharp mind but slower delivery, patient, faintly amused by everything"}
        },
        "lines": [
            "I've read every book in this library and I still have more questions than answers.",
            "The research consumed fifteen years of my life and I would not trade one day of it.",
            "Young scholars ask how things work. I spent my life asking why they matter."
        ]
    },
    "rogue": {
        "name": "Cael",
        "personality": "street-smart, pragmatic, darkly humorous",
        "ages": {
            "young": {"age": "15", "voice_mod": "scrappy, fast-talking street kid, slightly defiant, hiding vulnerability"},
            "adult": {"age": "34", "voice_mod": "smooth operator, sardonic, controlled, deceptively relaxed"},
            "elder": {"age": "68", "voice_mod": "raspy from years of hard living, wry humor intact, seen-it-all world-weariness"}
        },
        "lines": [
            "I'll get the coin. Don't worry about how — you never want to know how.",
            "I've stolen from kings and I've starved in gutters. Neither defines me.",
            "I've outlived six crime lords, three guild masters, and two people who swore they'd kill me. Lucky, I guess."
        ]
    }
}
print("Config loaded successfully.")

In [ ]:
# Clear memory and load model
gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
print(f"Loading model {model_id}...")
model = Qwen3TTSModel.from_pretrained(
    model_id,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)
print("Model loaded successfully!")

In [ ]:
ages_keys = ["young", "adult", "elder"]
sample_rate = 24000

for char_key, char in CHARACTERS.items():
    print(f"\n=== Generating Ages for {char['name']} ({char_key.upper()}) ===")
    for i, age_key in enumerate(ages_keys):
        age_data = char["ages"][age_key]
        line = char["lines"][i]
        
        instruct = f"{char['personality']}, {age_data['voice_mod']}, speaking at age {age_data['age']}"
        print(f"\nAge Stage: {age_key.upper()} (Age {age_data['age']})")
        print(f"Instruct: {instruct}")
        print(f"Line: \"{line}\"")
        
        res = model.generate_voice_design(
            text=line,
            language="English",
            instruct=instruct
        )
        audio_array, sr = res if isinstance(res, tuple) else (res, sample_rate)
        if hasattr(audio_array, "cpu"):
            audio_array = audio_array.cpu().numpy()
        audio_array = np.squeeze(audio_array)

        filename = f"aging_{char_key}_{age_key}.wav"
        filepath = os.path.join(OUTPUT_DIR, filename)
        sf.write(filepath, audio_array, sr)
        
        display(Audio(filepath))

In [ ]:
print("\n=== Life Story Montages ===")
print("This simulates a flashback or life-review cutscene.")
silence_1_5s = np.zeros(int(sample_rate * 1.5), dtype=np.float32)

for char_key, char in CHARACTERS.items():
    print(f"\nLife Montage: {char['name']}")
    combined_audio = []
    
    for age_key in ages_keys:
        fpath = os.path.join(OUTPUT_DIR, f"aging_{char_key}_{age_key}.wav")
        if os.path.exists(fpath):
            audio_data, sr = sf.read(fpath)
            combined_audio.extend(audio_data)
            combined_audio.extend(silence_1_5s)
            
    if combined_audio:
        combined_audio = np.array(combined_audio, dtype=np.float32)
        montage_path = os.path.join(OUTPUT_DIR, f"aging_{char_key}_life_story.wav")
        sf.write(montage_path, combined_audio, sample_rate)
        display(Audio(montage_path))

In [ ]:
print("\n=== Cross-Character Age Comparison (Elder) ===")
elder_combined = []
for char_key in CHARACTERS.keys():
    fpath = os.path.join(OUTPUT_DIR, f"aging_{char_key}_elder.wav")
    if os.path.exists(fpath):
        audio_data, sr = sf.read(fpath)
        elder_combined.extend(audio_data)
        elder_combined.extend(silence_1_5s)

if elder_combined:
    elder_combined = np.array(elder_combined, dtype=np.float32)
    elder_comp_path = os.path.join(OUTPUT_DIR, "aging_all_elders.wav")
    sf.write(elder_comp_path, elder_combined, sample_rate)
    display(Audio(elder_comp_path))

## 🎮 Game Engine Integration Notes
- **Syncing:** In Unity's Timeline or Godot's AnimationPlayer, you can schedule these voice clips to play consecutively with appropriate transition animations (e.g., crossfades, screen flashes).
- **Consistency Pipeline:** For production, it is highly recommended to use this `VoiceDesign` model to generate the "reference" audio for each age, and then switch to the `Qwen/Qwen3-TTS-12Hz-1.7B-Base` model using `generate_voice_clone(..., voice_clone_prompt=...)` to ensure maximum consistency across all game dialogue for that specific age.

In [ ]:
import shutil

# Zip the output directory for easy download
shutil.make_archive("/content/voice_aging_assets", 'zip', OUTPUT_DIR)
print("Assets zipped successfully to /content/voice_aging_assets.zip")

from google.colab import files
try:
    files.download("/content/voice_aging_assets.zip")
except:
    print("File download is only available in a live Colab environment.")